# Exercise — Adjudicate Master-Data Matches & Set Survivorship

**Trailhead Provisions** has duplicate customers because marketing keys on email while loyalty
keys on `customer_id`. Run the resolution workflow, adjudicate the **ambiguous** band, and
document survivorship. See `INSTRUCTIONS.md`.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import GovernedCatalog, apply_adjudications

gc = GovernedCatalog("trailhead.db")
m = gc.resolve_customers()
print("auto-merge:", int((m.pairs.band == "auto-merge").sum()),
      "| ambiguous:", len(m.ambiguous),
      "| rejected:", int((m.pairs.band == "reject").sum()))
m.ambiguous[["id_a", "name_a", "email_a", "id_b", "name_b", "email_b", "score"]]

## 1. Adjudicate each ambiguous pair
Decide `accept` (same person) or `reject` by row index. Inspect the pairs above before deciding.

In [ ]:
# All six are same-name + email-variant (domain swap, inserted digit, or
# whitespace name) of one person -> accept all six as merges.
decisions = {0: "accept", 1: "accept", 2: "accept",
             3: "accept", 4: "accept", 5: "accept"}
outcome = apply_adjudications(m, decisions)
print({k: v for k, v in outcome.items() if k != "merges"})
outcome["merges"]

## 2. Stewardship decision log
Replace the cell below with your log.

### Stewardship decision log

| Pair signal | Decision | Rationale |
|---|---|---|
| same name, domain-swapped email | **accept** | same person re-registered with a different mail provider |
| same name, inserted-digit email | **accept** | typo/auto-suffix on re-entry — clearly one person |
| whitespace name variant | **accept** | cosmetic name difference only |

**Survivorship rule:** the **lowest `customer_id` wins** as the golden record — loyalty is the
designated customer system of record and lower ids are the original registrations; non-key
attributes take the most-recent non-null value.

**False-positive vs false-negative:** I accept only pairs with *both* a name signal and an
email-local signal; pairs sharing only a last name stay unmerged. Under GDPR, over-merging (one
person seeing another's data) is worse than a missed duplicate. The durable fix is structural:
push the loyalty `customer_id` into marketing as the shared key, eliminating email-based
resolution entirely.